# OpenAI 빠른 시작

<h1 id="overview">개요</h1>

대규모 언어 모델(LLM)은 사용자가 입력한 텍스트를 바탕으로 다음에 올 가능성이 높은 텍스트를 생성하는 모델입니다. 단순히 문장을 이어 쓰는 것처럼 보이지만, 실제로는 학습 과정에서 문법, 지식, 코드 패턴, 대화 흐름, 요약 방식 같은 다양한 언어 패턴을 함께 익힙니다.

이 노트북은 Azure OpenAI를 처음 사용하는 수강생이 다음 흐름을 한 번에 경험하도록 구성되어 있습니다.

- APIM을 통해 Azure OpenAI에 안전하게 접속하기
- `.env`에서 endpoint, key, deployment 이름을 읽어오기
- `gpt-5.4-mini` 배포로 Chat Completions 호출하기
- 프롬프트 설계의 기본 개념 이해하기
- 요약, 분류(classification), 제품명 생성 같은 텍스트 작업 실습하기
- `text-embedding-3-large`로 임베딩을 만들고 유사도 비교하기

![LLM 개념](image/LLM_NLP.png)

## 목차

- <a href="#overview">개요</a>
- <a href="#azure-openai-service">Azure OpenAI 서비스 시작하기</a>
- <a href="#first-prompt">첫 번째 프롬프트 작성하기</a>
- <a href="#setup-credentials">1. 라이브러리 로드 및 APIM 자격 증명 설정</a>
- <a href="#model-selection">2. 적합한 모델 찾기</a>
- <a href="#prompt-design">3. 프롬프트 설계</a>
- <a href="#run-first-call">4. 실행</a>
- <a href="#use-cases">여러 사용 사례에 대한 연습</a>
  - <a href="#summarize-text">텍스트 요약</a>
  - <a href="#classify-text">텍스트 분류</a>
  - <a href="#generate-product-names">새로운 제품 이름 생성</a>
  - <a href="#embeddings">임베딩</a>
  - <a href="#cnn-dailymail">CNN DailyMail 데이터셋의 기사 비교</a>
  - <a href="#komantle">꼬맨틀</a>
- <a href="#references">참고자료</a>

<h3 id="azure-openai-service">Azure OpenAI 서비스 시작하기</h3>

일반적으로 Azure OpenAI를 직접 사용하려면 Azure 포털 또는 Azure AI Foundry에서 리소스를 만들고, 사용할 모델을 배포한 뒤, endpoint와 인증 정보를 애플리케이션에 설정해야 합니다.

이번 워크숍에서는 수강생이 개별 Azure OpenAI 리소스 키를 직접 사용하지 않습니다. 대신 강의용으로 미리 준비된 Azure AI Foundry 모델 배포를 APIM 공개 게이트웨이 뒤에 연결해 사용합니다.

이 구조의 장점은 다음과 같습니다.

- 수강생에게 AOAI 리소스 키를 직접 배포하지 않아도 됩니다.
- APIM 구독 키로 호출 권한을 관리할 수 있습니다.
- 노트북 코드는 Azure OpenAI SDK를 계속 사용하면서 endpoint만 APIM gateway로 바꿔 사용할 수 있습니다.

[Azure OpenAI 모델 문서](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)

<h3 id="first-prompt">첫 번째 프롬프트 작성하기</h3>

이 짧은 연습은 APIM을 통해 Azure OpenAI 모델에 첫 프롬프트를 보내는 과정입니다. 예제 질문은 단순하지만, 이후 모든 실습의 기본 구조가 여기에 들어 있습니다.

**단계**:  
1. Python 환경에 OpenAI 라이브러리를 설치합니다.  
2. `.env`에서 APIM 게이트웨이 URL과 APIM 구독 키를 로드합니다.  
3. 사용할 5.4 계열 모델 배포 이름을 선택합니다.  
4. 모델에게 전달할 사용자 프롬프트를 작성합니다.  
5. Chat Completions API를 호출하고 응답을 확인합니다.

처음에는 프롬프트와 응답의 구조를 보는 것이 중요합니다. 모델이 어떤 지시를 받았고, 어떤 형식으로 답했는지 확인하면서 다음 섹션의 프롬프트 설계 개념으로 이어집니다.

<h3 id="setup-credentials">1. 라이브러리 로드 및 APIM 자격 증명 설정</h3>

아래 코드 셀에서는 `.env` 파일을 읽고 Azure OpenAI SDK 클라이언트를 생성합니다. 이 워크숍에서는 실제 Azure OpenAI endpoint가 아니라 APIM gateway endpoint를 사용하므로, APIM 구독 키를 `Ocp-Apim-Subscription-Key` 헤더에도 함께 넣습니다.

In [2]:
import os
from pathlib import Path
from IPython.display import Markdown, display
from openai import AzureOpenAI
from dotenv import load_dotenv
dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")

if not azure_openai_endpoint or not azure_openai_key:
    raise ValueError(".env 파일에 AZURE_OPENAI_ENDPOINT와 AZURE_OPENAI_KEY를 설정하세요.")

client = AzureOpenAI(
  azure_endpoint=azure_openai_endpoint,
  api_key=azure_openai_key,
  api_version=azure_openai_api_version,
  default_headers={"Ocp-Apim-Subscription-Key": azure_openai_key},
)

print(f"Azure OpenAI endpoint: {azure_openai_endpoint}")
print(f"API version: {azure_openai_api_version}")

Azure OpenAI endpoint: https://apim-ai-workshop-010.azure-api.net/
API version: 2025-04-01-preview


<h3 id="model-selection">2. 적합한 모델 찾기</h3>

모델을 호출할 때 코드에는 실제 모델 이름이 아니라 **배포 이름(deployment name)** 을 전달합니다. Azure AI Foundry에서 모델을 배포할 때 정한 이름이 곧 API 호출에 사용하는 `model` 값이 됩니다.

이 워크숍에서는 `gpt-5.4-mini` 배포를 기본 LLM 배포로 사용합니다. 배포 이름은 코드에 직접 고정하지 않고 `.env`의 `AZURE_OPENAI_DEPLOYMENT_NAME`에서 읽어옵니다. 이렇게 하면 강의 환경에서 배포 이름이 바뀌어도 노트북 코드를 수정하지 않고 `.env`만 바꾸면 됩니다.

[Azure OpenAI 모델](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/concepts/models)

In [3]:
# .env에 설정된 5.4 계열 Chat Completions 배포를 사용합니다.
chat_model = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.4-mini")
print(f"Chat deployment: {chat_model}")

Chat deployment: gpt-5.4-mini


<h3 id="prompt-design">3. 프롬프트 설계</h3>

대규모 언어 모델은 사용자의 입력을 보고 다음에 올 가능성이 높은 텍스트를 생성합니다. 그래서 같은 모델을 쓰더라도 프롬프트를 어떻게 쓰느냐에 따라 답변의 정확도, 형식, 길이, 톤이 크게 달라집니다.

모델은 프롬프트를 통해 다음과 같은 능력을 발휘할 수 있습니다.

* 철자와 문법을 반영한 글쓰기
* 문장 요약과 패러프레이징
* 질문에 답하기
* 대화를 이어가기
* 여러 언어로 글쓰기
* 코드 작성과 설명
* 정해진 카테고리로 분류하기

#### 대규모 언어 모델 제어 방법

대규모 언어 모델의 입력 중 가장 영향력이 큰 요소는 프롬프트입니다. 같은 모델과 같은 배포를 사용하더라도 시스템 메시지와 사용자 메시지를 어떻게 구성하느냐에 따라 응답의 형식, 깊이, 톤이 달라질 수 있습니다.

대표적인 제어 방법은 다음과 같습니다.

- **명령(Instruction)**: 모델에게 원하는 작업을 직접 말합니다. 예: `다음 문장을 세 줄로 요약해줘.`
- **완성(Completion)**: 원하는 출력의 시작 부분을 제공하고 모델이 이어 쓰게 합니다. 예: `경복궁은 조선 시대의...`
- **데모(Demonstration)**: 모델이 따라 할 입력/출력 예시입니다. 예를 들어 `문장: 배송이 늦었어요. → 감정: 부정`처럼 원하는 패턴을 한 번 보여주는 것입니다.
- **Few-shot prompting**: 이런 데모를 프롬프트 안에 몇 개 넣어 모델이 분류 기준, 출력 형식, 말투를 스스로 따라 하게 만드는 방법입니다. 즉, 데모는 예시 하나이고 few-shot prompting은 그 예시들을 활용하는 프롬프트 작성 기법입니다.
- **미세 조정(Fine-tuning)**: 수백 개 이상의 입력/출력 예제로 모델을 추가 학습시켜 특정 업무 패턴을 더 안정적으로 따르게 합니다.

예를 들어 고객 문의를 `가격`, `하드웨어 지원`, `소프트웨어 지원` 중 하나로 나누는 작업은 **분류(classification)** 작업입니다. 이때 모델을 **분류기(classifier)**처럼 사용할 수 있습니다. 간단한 경우에는 프롬프트만으로 충분하지만, 기준이 복잡하거나 일관성이 매우 중요하면 few-shot 예시나 fine-tuning을 검토할 수 있습니다.

#### 프롬프트를 작성하는 세 가지 기본 지침

**보여주고 말하기**. 모델에게 원하는 작업을 말로 설명하고, 가능하면 예시까지 같이 보여주세요. 예를 들어 목록 정렬, 감정 분류, JSON 형식 출력처럼 원하는 형식이 분명한 작업은 예시를 함께 주면 결과가 더 안정적입니다.

**품질 데이터 제공**. 프롬프트에 넣는 예시는 모델이 따라 할 기준입니다. 라벨이 틀린 분류 예시, 오탈자가 있는 카테고리명, 서로 다른 출력 형식이 섞여 있으면 모델도 그 혼란을 따라 할 수 있습니다. 예시를 넣을 때는 입력과 출력 형식이 일관적인지 먼저 확인하세요.

**설정을 확인하세요.** `temperature`, `top_p`, `max_completion_tokens` 같은 파라미터는 응답 생성 방식에 영향을 줍니다. 제품 이름 생성처럼 다양한 후보가 필요한 작업은 `temperature`를 높여볼 수 있고, 분류처럼 일관성이 중요한 작업은 낮게 두는 편이 좋습니다. 다만 모델과 API 버전에 따라 지원되는 파라미터가 다를 수 있으므로, 필요한 경우 모델 문서를 확인합니다.

참고: [Prompt engineering techniques](https://learn.microsoft.com/azure/ai-services/openai/concepts/prompt-engineering)

<h3 id="run-first-call">4. 실행</h3>

이제 실제로 모델을 호출합니다. Chat Completions API는 `messages` 배열을 입력으로 받습니다. 여기서는 시스템 메시지로 모델의 기본 역할을 정하고, 사용자 메시지로 실제 질문을 전달합니다.

응답은 Markdown으로 렌더링해 줄바꿈, 목록, 굵게 표시가 자연스럽게 보이도록 출력합니다.

In [4]:
# 첫 번째 프롬프트를 한글로 작성합니다.
text_prompt = "경복궁은 어떤 곳인가요?"

response = client.chat.completions.create(
  model=chat_model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

display(Markdown(response.choices[0].message.content))

경복궁은 **조선 왕조의 법궁(정궁)**으로, 서울 한복판에 있는 대표적인 궁궐입니다.  
쉽게 말해, 조선 시대 왕이 공식적으로 정사를 보던 가장 중요한 궁궐이에요.

### 간단히 정리하면
- **건립**: 1395년, 조선 태조 때
- **위치**: 서울 종로구
- **의미**: “큰 복이 넓게 퍼지라”는 뜻의 이름
- **역할**: 왕의 생활 공간이자 국가 운영의 중심지

### 볼거리
- **근정전**: 왕이 공식 행사를 하던 곳
- **경회루**: 연회와 외교 행사가 열리던 아름다운 누각
- **강녕전**: 왕의 침전
- **교태전**: 왕비의 생활 공간
- **광화문**: 경복궁의 정문

### 역사적 의미
경복궁은 임진왜란 때 크게 소실되었고, 이후 오랫동안 복원되지 못하다가 조선 후기와 근대에 걸쳐 다시 정비되었습니다. 그래서 오늘날에는 **조선의 궁궐 문화와 한국 전통 건축을 대표하는 상징적인 장소**로 여겨집니다.

원하시면 제가 **경복궁을 처음 가는 사람 기준으로 관람 코스**도 추천해드릴게요.

### 동일한 호출을 반복하여 결과를 비교하세요

생성형 모델은 같은 질문에도 매번 완전히 동일한 답을 보장하지 않을 수 있습니다. 같은 프롬프트를 한 번 더 실행해 보고, 표현 방식이나 세부 내용이 어떻게 달라지는지 비교해보세요.

이 관찰은 뒤에서 다룰 `temperature` 같은 생성 파라미터를 이해하는 데 도움이 됩니다.

In [5]:
response = client.chat.completions.create(
  model=chat_model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

display(Markdown(response.choices[0].message.content))

경복궁은 **조선 왕조의 대표적인 궁궐**입니다.  
1395년(태조 4)에 한양에 처음 지어진 궁궐로, 조선의 법궁(가장 중심이 되는 궁궐)이었어요.

간단히 말하면:

- **조선의 왕이 살고 정사를 보던 곳**
- **왕실 의례와 국가 행사가 열리던 곳**
- 지금은 **서울의 대표적인 역사 유적지**이자 관광 명소

특징도 몇 가지 있어요:

- **광화문**: 경복궁의 정문
- **근정전**: 왕이 공식적인 일을 보던 중심 건물
- **경회루**: 연회와 행사가 열리던 연못 위 누각
- **향원정**: 아름다운 정원과 연못이 있는 곳

경복궁은 임진왜란 때 크게 소실되었고, 이후 오랫동안 복원 작업이 이어졌습니다.  
지금도 많은 사람들이 전통 건축과 한국 역사를 느끼기 위해 찾는 곳입니다.

원하시면 제가 **경복궁의 역사**, **대표 건물**, 또는 **관람 팁**까지 더 자세히 설명해드릴게요.

<h2 id="summarize-text">텍스트 요약</h2>

#### 도전 과제

긴 텍스트를 짧고 핵심적인 문장으로 줄이는 작업입니다. 요약은 LLM의 대표적인 활용 사례 중 하나이며, 회의록 정리, 기사 요약, 고객 문의 요약, 문서 검색 결과 요약 등에 자주 사용됩니다.

이 예제에서는 문단 끝에 `Tl;dr`을 붙여 모델이 앞의 내용을 요약하도록 유도합니다. `Tl;dr`은 “too long; didn't read”의 줄임말로, 긴 글을 짧게 요약해달라는 신호처럼 사용할 수 있습니다.

더 안정적인 요약을 원한다면 다음처럼 조건을 더 구체적으로 줄 수 있습니다.

- 세 문장 이내로 요약하기
- bullet point로 요약하기
- 초등학생도 이해할 수 있게 요약하기
- 핵심 결론만 한 줄로 요약하기

아래 코드 셀에서는 긴 문단을 `prompt` 변수에 넣고, 모델에게 요약하도록 요청합니다.

<h1 id="use-cases">여러 사용 사례에 대한 연습</h1>

이제 같은 Chat Completions 호출 구조를 사용해 여러 작업을 실습합니다. 중요한 점은 모델이나 API가 바뀌는 것이 아니라, **프롬프트를 어떻게 쓰느냐에 따라 작업의 성격이 달라진다**는 것입니다.

이번 섹션에서는 다음 작업을 다룹니다.

1. 텍스트 요약
2. 텍스트 분류
3. 새로운 제품 이름 생성
4. 임베딩 기반 유사도 비교

In [6]:
prompt = "대규모 텍스트 말뭉치에 대한 사전 학습 후 특정 작업에 대한 미세 조정을 통해 많은 NLP 작업과 벤치마크에서 상당한 이점을 입증한 바 있습니다. 이 방법은 일반적으로 아키텍처에서 작업에 구애받지 않지만, 여전히 수천 또는 수만 개의 예제로 구성된 작업별 미세 조정 데이터 세트가 필요합니다. 반면, 인간은 일반적으로 몇 가지 예제나 간단한 명령어만으로 새로운 언어 작업을 수행할 수 있지만, 현재의 NLP 시스템에서는 여전히 많은 어려움을 겪고 있습니다. 여기에서는 언어 모델을 확장하면 작업에 구애받지 않고 소수의 예제만으로도 성능이 크게 향상되며, 때로는 이전의 최첨단 미세 조정 접근 방식에 비해 경쟁 우위에 도달할 수도 있음을 보여줍니다.\n\nTl;dr"


In [7]:
response = client.chat.completions.create(
  model=chat_model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

display(Markdown(response.choices[0].message.content))

이 글의 핵심은 다음과 같습니다:

**언어 모델을 매우 크게 키우면, 별도의 작업별 미세 조정 없이도 몇 개의 예시만 보고 새로운 NLP 작업을 꽤 잘 수행할 수 있다**는 것입니다.  
즉, **few-shot 학습 능력은 모델 규모가 커질수록 크게 좋아지며**, 경우에 따라 기존의 **작업별 미세 조정 방식과 비슷한 성능**까지 낼 수 있다는 주장입니다.

한 줄 요약:  
**“큰 언어 모델은 적은 예시만으로도 새 작업을 잘한다.”**

<h2 id="classify-text">텍스트 분류</h2>

#### 도전 과제

분류(classification)는 입력을 정해진 카테고리 중 하나로 나누는 작업입니다. 이 작업을 수행하는 모델이나 시스템을 분류기(classifier)라고 부릅니다.

예를 들어 고객 문의가 들어왔을 때 이를 `가격`, `하드웨어 지원`, `소프트웨어 지원` 중 하나로 나눌 수 있습니다. 이런 분류는 고객 지원 티켓 라우팅, 리뷰 감정 분석, 문서 주제 분류, 스팸 탐지 등에 활용됩니다.

이 예제에서는 별도의 분류 모델을 학습하지 않고, LLM에게 카테고리 목록과 고객 문의를 함께 전달해 분류기처럼 사용합니다.

고객 문의: 안녕하세요, 제 노트북 키보드의 키 하나가 최근에 부러져서 교체가 필요합니다.

분류된 카테고리:

In [8]:
prompt = "다음 문의를 다음 중 하나의 카테고리로 분류합니다: [가격, 하드웨어 지원, 소프트웨어 지원]\n\n문의: 안녕하세요, 노트북 키보드의 키 중 하나가 최근에 고장 나서 교체해야 합니다.\n\n분류된 카테고리:"
print(prompt)

다음 문의를 다음 중 하나의 카테고리로 분류합니다: [가격, 하드웨어 지원, 소프트웨어 지원]

문의: 안녕하세요, 노트북 키보드의 키 중 하나가 최근에 고장 나서 교체해야 합니다.

분류된 카테고리:


In [9]:
response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {"role":"system", "content":"You are a helpful assistant."},
        {"role":"user", "content":prompt},
    ],
)

display(Markdown(response.choices[0].message.content))

하드웨어 지원

<h2 id="generate-product-names">새로운 제품 이름 생성</h2>

#### 도전 과제

이번 예제는 제품 설명과 몇 가지 예시 이름을 프롬프트에 넣고, 모델이 같은 패턴으로 새로운 제품 이름을 이어서 생성하도록 하는 실습입니다.

이 방식은 few-shot prompting의 간단한 예입니다. 모델에게 “이런 입력에는 이런 출력이 나온다”는 예시를 보여주면, 모델은 다음 입력에도 같은 형식을 적용하려고 합니다.

제품 이름 생성처럼 다양한 후보가 필요한 작업에서는 `temperature` 값을 높여볼 수 있습니다. 이 노트북에서는 `temperature=1.2`를 사용해 응답의 다양성을 높입니다. 반대로 분류처럼 일관성이 중요한 작업에서는 낮은 값을 사용하는 편이 좋습니다.

제품 설명: 가정용 밀크셰이크 제조기  
시드 단어: 빠른, 건강한, 컴팩트  
제품 이름: HomeShaker, Fit Shaker, QuickShake, Shake Maker

제품 설명: 모든 발 크기에 맞는 신발 한 켤레.
시드 단어: 적응 가능한, 맞춤형, 옴니핏.

In [10]:
prompt = "제품 설명: 가정용 밀크셰이크 메이커\n시드워드: 빠르고, 건강하고, 컴팩트한 제품입니다.\n제품 이름: 홈셰이커, 핏 셰이커, 퀵셰이크, 셰이크 메이커\n\n제품 설명: 모든 발 사이즈에 맞는 신발\n씨드워드: 적응성, 핏, 옴니핏.\n제품이름: "

print(prompt)

제품 설명: 가정용 밀크셰이크 메이커
시드워드: 빠르고, 건강하고, 컴팩트한 제품입니다.
제품 이름: 홈셰이커, 핏 셰이커, 퀵셰이크, 셰이크 메이커

제품 설명: 모든 발 사이즈에 맞는 신발
씨드워드: 적응성, 핏, 옴니핏.
제품이름: 


In [11]:
# 제품명 생성처럼 다양한 후보가 필요한 작업에서는 temperature를 높여볼 수 있습니다.
response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {"role":"system", "content":"You are a helpful assistant."},
        {"role":"user", "content":prompt},
    ],
    temperature=1.2,
)

display(Markdown(response.choices[0].message.content))

옴니핏, 핏슈, 애드핏, 올핏슈

원하시면 같은 톤으로 20개 더 만들어드릴게요.

<h2 id="embeddings">임베딩</h2>

임베딩은 텍스트를 숫자 벡터로 변환한 표현입니다. 사람이 보기에는 단어와 문장이 문자열이지만, 모델이 비교하고 계산하려면 숫자 형태가 필요합니다.

의미가 비슷한 텍스트는 임베딩 벡터 공간에서도 가까운 위치에 놓이는 경향이 있습니다. 이 성질을 활용하면 다음과 같은 작업을 만들 수 있습니다.

- 비슷한 문서 찾기
- 검색 결과 순위 매기기
- 추천 시스템 만들기
- 고객 문의를 비슷한 유형끼리 묶기
- 단어 의미 유사도 게임 만들기

이 워크숍에서는 `text-embedding-3-large` 배포를 사용해 단어, 문장, 기사 문서 간의 의미적 유사도를 계산합니다.

![Embedding](image/Embedding.png)

### 임베딩 모델 선택

이 워크숍에서는 `text-embedding-3-large` 배포를 임베딩 모델로 사용합니다. 이 모델은 텍스트를 최대 3072차원의 벡터로 변환합니다.

배포 이름은 코드에 직접 고정하지 않고 `.env`의 `EMBEDDING_MODEL_NAME`에서 읽어옵니다.

```python
model = os.getenv("EMBEDDING_MODEL_NAME", "text-embedding-3-large")
```

벡터 차원은 `.env`의 `AZURE_OPENAI_EMBEDDING_DIMENSIONS=3072`로 관리합니다. 차원 수가 클수록 더 많은 정보를 담을 수 있지만, 저장 공간과 계산 비용도 늘어납니다.

In [12]:
import numpy as np

embedding_model = os.getenv("EMBEDDING_MODEL_NAME", "text-embedding-3-large")
embedding_dimensions = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "3072"))
print(f"Embedding deployment: {embedding_model} ({embedding_dimensions} dimensions)")

def cosine_similarity(query_embedding, embeddings, distance_metric='cosine'):
    if distance_metric == 'cosine':
        distances = np.dot(embeddings, query_embedding) / (np.linalg.norm(embeddings) * np.linalg.norm(query_embedding))
        distances = 1 - distances  
    else:
        raise ValueError("Unsupported distance metric. Use 'cosine'.")

    return distances

Embedding deployment: text-embedding-3-large (3072 dimensions)


In [13]:
text = '게으른 개를 뛰어넘은 재빠른 갈색 여우'
client.embeddings.create(input=[text], model=embedding_model).data[0].embedding


[-0.0067138671875,
 0.005916595458984375,
 0.00044155120849609375,
 -0.02130126953125,
 0.041107177734375,
 0.01043701171875,
 -0.01971435546875,
 -0.0048370361328125,
 -0.03314208984375,
 -0.0303802490234375,
 -0.05816650390625,
 -0.000213623046875,
 0.02410888671875,
 -0.037384033203125,
 -0.003589630126953125,
 0.050689697265625,
 -0.0296478271484375,
 -0.00421142578125,
 -0.0074615478515625,
 -0.007656097412109375,
 0.0206146240234375,
 -0.02813720703125,
 0.048858642578125,
 -0.01187896728515625,
 -0.0240631103515625,
 0.023712158203125,
 0.003265380859375,
 0.015777587890625,
 -0.00667572021484375,
 0.005702972412109375,
 0.01552581787109375,
 0.03857421875,
 -0.0174713134765625,
 0.020843505859375,
 -0.004077911376953125,
 0.015838623046875,
 0.037750244140625,
 0.0248870849609375,
 0.0155181884765625,
 -0.013580322265625,
 -0.013885498046875,
 0.021453857421875,
 -0.03680419921875,
 -0.025299072265625,
 -0.01398468017578125,
 0.028045654296875,
 0.005252838134765625,
 -0.003850

In [14]:
# compare several words
automobile_embedding    = client.embeddings.create(input='자동차', model=embedding_model).data[0].embedding
vehicle_embedding       = client.embeddings.create(input='차량', model=embedding_model).data[0].embedding
dinosaur_embedding      = client.embeddings.create(input='공룡', model=embedding_model).data[0].embedding
stick_embedding         = client.embeddings.create(input='스틱', model=embedding_model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

0.39889864781896767
0.7221947755823874
0.7343865080519378


<h2 id="cnn-dailymail">CNN DailyMail 데이터셋의 기사 비교</h2>

이번 예제에서는 짧은 단어가 아니라 긴 기사 문서를 임베딩으로 변환해 비교합니다. 문서 검색이나 RAG 시스템에서는 사용자의 질문과 문서의 임베딩을 비교해 관련성이 높은 문서를 찾습니다.

여기서는 CNN/DailyMail 형식의 번역된 기사 예시를 사용해 기사 1번이 기사 2번, 기사 3번과 의미적으로 얼마나 가까운지 계산합니다.

In [15]:
import pandas as pd

# 번역된 기사와 하이라이트

cnn_daily_articles = [
    "브레멘, 독일 -- 2004년 FC 포르투가 모나코를 꺾고 챔피언스리그 결승에서 우승할 때 골을 넣었던 카를로스 알베르토가 분데스리가 클럽 베르더 브레멘에 구단 역대 최고 이적료인 780만 유로(1,070만 달러)에 합류했습니다. 카를로스 알베르토는 조제 무리뉴 감독 아래 FC 포르투에서 성공을 거뒀습니다. '나는 베르더와 함께 우승하기 위해 여기 왔습니다.'라고 22세의 그는 새 클럽에서 첫 훈련을 마친 뒤 말했습니다. '나는 브레멘이 마음에 들고, 오직 여기만 오고 싶었습니다.' 카를로스 알베르토는 플루미넨시에서 커리어를 시작해 2002년 캄페오나토 카리오카 우승을 도왔습니다. 2004년 1월에는 조제 무리뉴 감독이 이끌던 FC 포르투로 이적해 포르투갈 리그와 챔피언스리그 우승을 차지했습니다. 2005년 초에는 코린치안스로 이적해 브라질 세리 A 우승에 기여했으나, 2006년 코린치안스가 부진하자 에메르손 레앙 감독과 불화가 생겼습니다. 이들의 관계는 코파 수다메리카나에서 클럽 아틀레티코 라누스와의 경기에서 절정에 달했고, 카를로스 알베르토는 레앙 감독이 있는 한 다시는 코린치안스에서 뛰지 않겠다고 선언했습니다. 올해 1월부터는 친정팀 플루미넨시에서 임대 생활을 했습니다. 분데스리가 챔피언인 슈투트가르트는 일요일 레알 사라고사에서 에베르톤을 임대로 영입할 것이라고 밝혔습니다. 에베르톤은 2001~2005년 보루시아 도르트문트에서 활약한 바 있습니다. 금요일에는 2004년 분데스리가 득점왕이었던 아일톤이 레드스타 베오그라드에서 뒤스부르크로 1년 계약을 맺고 독일로 복귀했습니다. 친구에게 이메일 보내기.",
    "(CNN) -- 축구 슈퍼스타, 셀러브리티, 패션 아이콘, 수백만 달러의 인기남. 이제 데이비드 베컴이 미국 메이저리그 사커에서 활약하기 위해 할리우드 힐스로 향합니다. CNN은 베컴이 맨체스터 유나이티드에서 뛰는 꿈을 어떻게 이뤘는지, 그리고 잉글랜드 대표팀에서의 시간을 조명합니다. 세계적으로 유명한 축구선수 베컴은 LA 갤럭시와 5년 계약을 맺었고, 금요일에는 기자회견을 열고 새로운 등번호를 공개할 예정입니다. 이번 주, CNN의 '벡스' 베키 앤더슨이 베컴의 축구선수, 패션 아이콘, 글로벌 현상으로서의 삶을 심층적으로 살펴봅니다. 동런던 거리에서 할리우드 힐스까지, 베컴의 놀라운 성공 여정을 따라갑니다. 그녀는 미국 스포츠/연예계에서 가장 뜨거운 인물인 베컴의 진면목을 파헤치고, 그를 움직이게 하는 동기와 '황금발'의 비밀을 탐구합니다. CNN은 맨유에서의 꿈, 팝스타 빅토리아와의 결혼, 잉글랜드 대표팀에서의 고난과 영광, 레알 마드리드 이적, 그리고 이제 LA 홈디포 스타디움까지 베컴의 인생을 돌아봅니다. 베컴과 가족이 LA 생활에 어떻게 적응할지, 현지인들과 명소, 셀럽 문화 적응기를 다룹니다. 베컴은 이미 광고, 게임, 패션 등 다양한 분야에서 얼굴을 알렸습니다. 미국에서 축구가 '여자아이들만의 스포츠'라는 인식이 바뀌고 있으며, 점점 더 많은 아이들이 유럽 축구를 선택하고 있습니다. CNN은 미국에서 활약한 해외 스타들의 영향과 현재의 변화를 살펴보고, LA의 데이비드 베컴 아카데미도 조명합니다. 친구에게 이메일 보내기.",
    "로스앤젤레스, 캘리포니아(CNN) -- 이라크에서 화상을 입은 5살 소년 유시프가 유니버설 스튜디오에서 코너를 돌자마자 가장 좋아하는 슈퍼히어로를 만났습니다. 유시프는 스파이더맨의 열렬한 팬이었습니다. '가장 좋았어요.'라고 그는 말했습니다. 스파이더맨은 다른 슈퍼히어로들과 함께 사륜 오토바이를 타고 등장했습니다. 스파이더맨은 유시프에게 다가와 인사를 건네고, 상징적인 파란색과 빨간색 타이즈로 소년을 부드럽게 안아주었습니다. 그는 유시프에게 손목에서 거미줄 쏘는 법 등 몇 가지 묘기를 보여줬습니다. 이번엔 진짜 거미줄은 나오지 않았지만요. '좋았어, 유시프!' 스파이더맨이 소년이 손동작을 따라하자 말했습니다. 다른 슈퍼히어로들도 몰려와 구경했습니다. 그린 고블린도 소년에게 인사를 건넸습니다. 유시프는 악당에게는 별로 관심이 없었습니다. 스파이더맨이 최고였죠. '가장 좋았어요.'라고 소년은 나중에 다시 말했습니다. '다시 만나고 싶어요.' 그리고 덧붙였습니다. '진짜 스파이더맨이 아니라는 건 알아요.' 이날은 소년의 악몽이 잠시 잊힌 꿈같은 하루였습니다. 그는 스폰지밥, 래시, 3살 오랑우탄 아치도 만났습니다. 아치는 유시프의 손을 잡고 놓지 않았습니다. 유시프가 손을 빼도 다시 잡으려 했습니다. 유시프는 놀이방에서 스펀지볼을 쏘며 깔깔 웃었습니다. 바그다드에서 보던 무기와는 전혀 다른 장난감이었습니다. 그는 트램을 타고 유니버설 스튜디오의 백스테이지도 돌았습니다. 한순간 차가 흔들리고, 불과 연기가 피어오르고, 트럭이 돌진해왔지만 가족은 무사했습니다. '난 무서웠다.' 아빠가 말했습니다. '난 안 무서웠어요.' 유시프가 대답했습니다. 부모님은 하루 종일 미소를 지었습니다. 유시프는 14개월 된 여동생 아야를 유모차에 태워 밀었습니다. '우리가 여기 오고 싶었는지 물어볼 필요가 있었나요?' 아빠는 감탄했습니다. '결혼식 빼고 오늘이 내 인생에서 가장 행복한 날이에요.' 하루 전, 부모는 이라크에서 미국으로 오게 된 사연과 9개월 전 복면을 쓴 남자들이 집 앞에서 아들을 납치해 불을 지른 일을 이야기했습니다. 엄마는 집 안에서 아들의 비명을 들었고, 아빠는 바그다드 전역을 돌며 도움을 구했지만 아무도 도와주지 않았습니다. 두 달간의 입원, 마취도 없이 치료받는 아들의 비명을 병원 밖에서 들었습니다. CNN에 사연을 알리는 것이 가족의 생명을 위협할 수 있다는 걸 알았지만, 아들을 위해서라면 뭐든지 하겠다고 했습니다. '이라크는 끝났다.' 아빠는 영어로 말했습니다. 아랍어로는 조국이 이런 자유를 누릴 수 없을 거라고 덧붙였습니다. 너무 많은 폭력과 살인 때문입니다. 두 아이 모두 전쟁만 보고 자랐지만, 이번 주 미국에서의 삶은 전혀 달랐습니다. '꿈만 같아요.' 아빠는 말했습니다. 그는 바그다드에서 자원봉사를 많이 했다고 했습니다. '아마 그래서 지금 도움을 받는 것 같아요.' 유니버설 스튜디오에서 아빠는 계곡을 내려다보며 '좋은 미국, 좋은 미국'이라고 영어로 말했습니다. 친구에게 이메일 보내기. CNN의 아르와 데이먼이 이 보도에 기여했습니다."
]

cnn_daily_article_highlights = [
    "베르더 브레멘, 카를로스 알베르토 영입에 구단 최고 이적료 1,070만 달러 지불.\n브라질 미드필더, 2004년 FC 포르투와 챔피언스리그 우승.\n올해 1월부터 친정팀 플루미넨시에서 임대 생활.",
    "베컴, LA 갤럭시와 5년 계약 체결.\n새 계약은 2007년 7월 1일부터 발효.\n전 잉글랜드 주장, 금요일 기자회견 및 새 등번호 공개 예정.\nCNN, 축구선수·패션 아이콘·글로벌 현상으로서의 베컴 조명.",
    '소년, 스파이더맨 만남에 "가장 좋았어요"\n유시프, 유니버설 스튜디오에서 스폰지밥·래시·오랑우탄도 만남.\n아빠: "결혼식 빼고 오늘이 내 인생에서 가장 행복한 날"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()
                   

,articles,highligths
0,"브레멘, 독일 -- 2004년 FC 포르투가 모나코를 꺾고 챔피언스리그 결승에서 우...","베르더 브레멘, 카를로스 알베르토 영입에 구단 최고 이적료 1,070만 달러 지불...."
1,"(CNN) -- 축구 슈퍼스타, 셀러브리티, 패션 아이콘, 수백만 달러의 인기남. ...","베컴, LA 갤럭시와 5년 계약 체결.\n새 계약은 2007년 7월 1일부터 발효...."
2,"로스앤젤레스, 캘리포니아(CNN) -- 이라크에서 화상을 입은 5살 소년 유시프가 ...","소년, 스파이더맨 만남에 ""가장 좋았어요""\n유시프, 유니버설 스튜디오에서 스폰지밥..."


In [16]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=embedding_model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=embedding_model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=embedding_model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

0.608792527594137
0.8586036594349853


<h2 id="komantle">꼬맨틀</h2>

https://semantle-ko.newsjel.ly/

꼬맨틀은 정답 단어와 사용자가 입력한 단어의 의미적 유사도를 이용하는 게임입니다. 사용자가 입력한 단어가 정답과 의미적으로 가까울수록 더 높은 점수를 받을 수 있습니다.

이런 게임은 임베딩의 좋은 응용 예입니다. 단어를 임베딩 벡터로 바꾸고, 정답 단어 벡터와의 거리를 계산하면 “의미적으로 얼마나 가까운지”를 숫자로 표현할 수 있습니다.

<h1 id="references">참고자료</h1>

추가로 학습할 때 참고할 만한 문서입니다.

- [Azure OpenAI Service documentation](https://learn.microsoft.com/azure/ai-services/openai/)
- [Azure OpenAI models](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)
- [Azure OpenAI prompt engineering](https://learn.microsoft.com/azure/ai-services/openai/concepts/prompt-engineering)
- [OpenAI Cookbook](https://github.com/openai/openai-cookbook)